<!-- nav -->
[← 7 · One constraint, many languages](07_One_Constraint_Many_Languages.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)

# 8 · Constraints for firmware and hardware

**The problem.** The logger's firmware, in C, checks each packet before sending it, and its FPGA's verification
engineers check the same condition with SystemVerilog assertions. In C, `uint8_t + uint8_t` is an `int`, and signed
overflow is undefined. In SystemVerilog, a bit can be `x`, unknown, or `z`, undriven. A constraint that is right in
Python can be wrong in either.

The Ccpp and SystemVerilog dialects model those languages' rules exactly, so you can translate a packet's constraint,
written once in Basic, and evaluate it the way the firmware and the hardware will. A Basic term carries no width: its
ints are unbounded, and what C's and SystemVerilog's types make of a value is theirs. Giving Basic's terms domains,
`int16` and `uint8`, waits for type inference, a later capability.

In [1]:
from mbse.Expressions import Evaluators, Expressions as E, Translators
from mbse.Expressions.Dialects import Ccpp, SystemVerilog
from toolkit import cold_chain

store = cold_chain()  # with Ccpp's and SystemVerilog's kinds installed
Expression, Term = store.Expressions.Framework.Expression, store.Expressions.Framework.Term
r, s = E.variable("raw"), E.variable("status")
with store.transaction():
    packet_ok = Expression(lambda e: e.root(r.ge(200).and_(r.le(800)).and_(s.bitand(0b100).eq(0))))
to_c = Translators.between(E.DIALECT, Ccpp.Expressions.DIALECT)
to_sv = Translators.between(E.DIALECT, SystemVerilog.Expressions.DIALECT)

## Step 1: C

A translation is a writer of the target's term, written in a transaction like any other:

In [2]:
with store.transaction():
    c = Term(to_c.forward(packet_ok))
print(Ccpp.Text.ToText(c))
for status in (0, 0b100):
    print(status, Ccpp.Evaluators.OfAny(c, {"raw": 523, "status": status}))

raw >= 200 && raw <= 800 && (status & 4) == 0
0 True
4 False


## Step 2: where C differs

C's integers have widths: an `int` is 32 bits. Basic's are unbounded, so a sum Basic computes exactly can overflow in
C, and signed overflow is undefined behavior in C: a compiler may assume it never happens. The evaluator raises instead
of picking a result, which is what a test should do:

In [3]:
with store.transaction():
    increment = Term(r.add(1))
    increment_c = Term(to_c.forward(increment))
print(Ccpp.Text.ToText(increment_c), "in Basic:", Evaluators.OfAny(increment, {"raw": 2147483647}))
try:
    Ccpp.Evaluators.OfAny(increment_c, {"raw": 2147483647})
except OverflowError as error:
    print("C:", error)

raw + 1 in Basic: 2147483648
C: signed overflow in +: 2147483648 does not fit 'int'


C's own types are C's: a `uint8_t` is promoted to `int` before it is added, so the sum is 300, not 44. Basic has no
counterpart for a constant of its own type yet, since its domain waits for type inference, and the translator refuses
it, naming it:

In [4]:
C = Ccpp.Expressions
with store.transaction():
    total = Term(C.binary("+", C.constant(200, "uint8_t"), C.constant(100, "uint8_t")))
print(Ccpp.Text.ToText(total), "=", Ccpp.Evaluators.OfAny(total, {}))
try:
    to_c.backward(total)
except ValueError as error:
    print(error)

(uint8_t)200 + (uint8_t)100 = 300
Ccpp constant 200 of uint8_t has no Basic counterpart


## Step 3: SystemVerilog

SystemVerilog's integers are 4-state vectors. Its evaluator gives a 4-state value, written as SystemVerilog would
display it:

In [5]:
with store.transaction():
    sv = Term(to_sv.forward(packet_ok))
print(SystemVerilog.Text.ToText(sv))
SystemVerilog.Evaluators.OfAny(sv, {"raw": 523, "status": 0})

raw >= 200 && raw <= 800 && (status & 4) == 0


1'b1

## Step 4: x and z

In simulation, a register that was never reset holds `x`. Whether that matters depends on the bit: the mask
looks only at bit 2, so an `x` elsewhere doesn't change the answer, but an `x` in bit 2 makes the whole condition `x`,
which an assertion reports as a failure to know. `Logic.of` makes a 4-state value from its bits:

In [6]:
Logic = SystemVerilog.Domains.Logic
for bits in ("00000000", "00000100", "000000x0", "00000x00"):
    print(bits, SystemVerilog.Evaluators.OfAny(sv, {"raw": 523, "status": Logic.of(bits)}))

00000000 1'b1
00000100 1'b0
000000x0 1'b1
00000x00 1'bx


This is Kleene's unknown again, in hardware's terms: `x` where the data doesn't decide.

## Step 5: what has no counterpart

`has` asks whether a property is present. A C struct or a SystemVerilog register always has its fields, so there's
nothing to translate it to, and the translator refuses, naming the term:

In [7]:
with store.transaction():
    reported = Term(E.variable("shipment").has("celsius"))
for translator in (to_c, to_sv):
    try:
        translator.forward(reported)
    except ValueError as error:
        print(error)

Basic call 'has' has no Ccpp counterpart
Basic call 'has' has no SystemVerilog counterpart


## Why it works this way

- **Model the language, don't approximate it.** Firmware and hardware bugs live in exactly the corners where languages
  differ: promotions, wrapping, undefined behavior, `x`. An evaluator that follows C's or IEEE 1800's rules finds them
  in a test, before the hardware does.
- **The neutral constraint stays neutral.** Basic doesn't take C's promotions or SystemVerilog's sizing. The translator
  writes them out, and the target's evaluator applies them.
- **Refuse what can't be said.** A constraint about whether data is present has no meaning for a fixed-layout packet; a
  silent approximation would be a bug in the translation.

That's the series: a constraint written once, as data, checked, stored, analyzed, specialized, and checked in each
language its users work in.

<!-- nav -->
[← 7 · One constraint, many languages](07_One_Constraint_Many_Languages.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)